# R21 Resting-State Randomise Results

Interactively inspect **cluster-extent corrected** randomise results on MNI anatomy and summarize the corresponding dual-regression stage-2 beta for sham, RTPJ, VLPFC, and BOTH. TFCE results are intentionally excluded.

## 1. Install notebook dependencies

This cell installs only packages missing from the active kernel. It can be rerun safely and does not require Neurodesk.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'code' / 'check_randomise_results.py').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter from the r21-rest repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
REQUIREMENTS = PROJECT_ROOT / 'notebooks' / 'requirements.txt'
required_imports = ('ipywidgets', 'matplotlib', 'nibabel', 'nilearn', 'numpy', 'pandas')
missing = [name for name in required_imports if importlib.util.find_spec(name) is None]
if missing:
    print('Installing missing notebook packages:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REQUIREMENTS)])
else:
    print('Notebook dependencies are already installed.')
print('Project root:', PROJECT_ROOT)

## 2. Load significant cluster-extent results

FSL corrected-p images contain `1-p`; the notebook defines the displayed region as voxels with `1-p > 0.95`.

In [ ]:
from functools import lru_cache
import warnings

import ipywidgets as widgets
import matplotlib.pyplot as plt
import nibabel as nib
from nilearn import datasets, image, plotting
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

CORRP_THRESHOLD = 0.95
CONDITION_ORDER = ['sham', 'rtpj', 'vlpfc', 'both']
CONDITION_LABELS = ['Sham', 'RTPJ', 'VLPFC', 'BOTH']
SUMMARY = PROJECT_ROOT / 'derivatives' / 'fsl' / 'randomise_summary' / 'task-rest_randomise_peak_summary.tsv'
if not SUMMARY.is_file():
    raise FileNotFoundError(f'Run code/check_randomise_results.py first: {SUMMARY}')

results = pd.read_csv(SUMMARY, sep='\t', dtype=str).fillna('')
significant = results.loc[(results['inference'] == 'cluster-extent') & (results['peak_gt_threshold'] == 'true') & (results['status'] == 'ok')].copy()
contrast_terms = {
    'both-minus-sham': ('BOTH', 'sham'),
    'both-minus-rtpj': ('BOTH', 'RTPJ'),
    'both-minus-vlpfc': ('BOTH', 'VLPFC'),
    'rtpj-minus-vlpfc': ('RTPJ', 'VLPFC'),
    'rtpj-minus-sham': ('RTPJ', 'sham'),
    'vlpfc-minus-sham': ('VLPFC', 'sham'),
    'both-minus-mean-rtpj-vlpfc': ('BOTH', 'average(RTPJ, VLPFC)'),
}
def signed_effect(row):
    first, second = contrast_terms[row['condition_contrast']]
    if row['direction'] == 'negative':
        first, second = second, first
    return f'{first} > {second}'

significant['effect'] = significant.apply(signed_effect, axis=1)
if significant.empty:
    raise RuntimeError('The summary contains no complete cluster-extent maps with peak 1-p > 0.95.')
print(f'Cluster-extent results available: {len(significant)}')
display(significant[['analysis', 'network', 'component', 'effect', 'peak_corrp']])

## 3. Define image and ROI-summary helpers

In [ ]:
def analysis_label(analysis):
    return {'0': 'denoised_dim-00_task-rest', '20': 'denoised_dim-20_task-rest', 'smith09': 'smith09_denoised'}[str(analysis)]

def project_path(value):
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path

def result_image(row):
    copied = str(row.get('copied_image', '')).strip()
    source = copied or str(row['corrp_file']).strip()
    path = project_path(source)
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

def result_label(row):
    analysis = {'0': 'ICA dim=auto', '20': 'ICA dim=20', 'smith09': 'direct Smith09'}[str(row['analysis'])]
    return f"{analysis} | {row['network']} map {int(row['component'])} | {row['effect']}"

@lru_cache(maxsize=None)
def extract_condition_values(row_index):
    row = significant.loc[row_index]
    corrp_img = nib.load(str(result_image(row)))
    corrp_data = np.asarray(corrp_img.dataobj)
    roi = np.isfinite(corrp_data) & (corrp_data > CORRP_THRESHOLD)
    if not roi.any():
        raise ValueError('Selected corrected-p map has no voxels above the threshold.')

    dr_dir = PROJECT_ROOT / 'derivatives' / 'fsl' / f"dual-regression_{analysis_label(row['analysis'])}.dr"
    order_file = dr_dir / 'input_order.tsv'
    order = pd.read_csv(order_file, sep='\t', dtype=str)
    component_index = int(row['component']) - 1
    records = []
    for item in order.itertuples(index=False):
        stage2 = dr_dir / f"dr_stage2_{item.dual_regression_label}.nii.gz"
        stage2_img = nib.load(str(stage2))
        if stage2_img.shape[:3] != corrp_img.shape[:3] or not np.allclose(stage2_img.affine, corrp_img.affine, atol=1e-3):
            raise ValueError(f'Grid mismatch between {stage2.name} and {result_image(row).name}')
        if len(stage2_img.shape) != 4 or component_index >= stage2_img.shape[3]:
            raise ValueError(f'Component {component_index + 1} is absent from {stage2.name}')
        component_data = np.asarray(stage2_img.dataobj[..., component_index])
        records.append({'participant': item.participant, 'condition': str(item.condition).lower(), 'stage2_beta': float(np.nanmean(component_data[roi]))})

    values = pd.DataFrame.from_records(records)
    counts = values.groupby('participant')['condition'].nunique()
    complete = counts[counts == 4].index
    values = values.loc[values['participant'].isin(complete)].copy()
    values['condition'] = pd.Categorical(values['condition'], CONDITION_ORDER, ordered=True)
    return values, corrp_img, roi

def plot_condition_bars(values, title):
    summary = values.groupby('condition', observed=False)['stage2_beta'].agg(['mean', 'sem', 'count']).reindex(CONDITION_ORDER)
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    colors = ['#8A8F98', '#2878B5', '#D95F59', '#3A9D6F']
    ax.bar(CONDITION_LABELS, summary['mean'], yerr=summary['sem'], capsize=5, color=colors, edgecolor='#222222', linewidth=0.8)
    ax.axhline(0, color='#555555', linewidth=0.8)
    ax.set_ylabel('Mean dual-regression stage-2 beta')
    ax.set_title(title, fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    display(summary.rename_axis('condition'))
    return summary

## 4. Select and inspect a result

Use the dropdown to switch analyses. The brain viewer is interactive: drag to rotate/cut through the image and use its controls to change the view.

In [ ]:
options = [(result_label(row), index) for index, row in significant.iterrows()]
selector = widgets.Dropdown(options=options, description='Result:', layout=widgets.Layout(width='95%'))
output = widgets.Output()

def render_result(change=None):
    with output:
        output.clear_output(wait=True)
        row_index = selector.value
        row = significant.loc[row_index]
        values, corrp_img, roi = extract_condition_values(row_index)
        thresholded = image.new_img_like(corrp_img, np.where(roi, np.asarray(corrp_img.dataobj), 0.0), copy_header=True)
        title = result_label(row)
        print(f'Suprathreshold voxels: {int(roi.sum())}; complete participants: {values.participant.nunique()}')
        viewer = plotting.view_img(thresholded, bg_img=datasets.load_mni152_template(resolution=2), threshold=CORRP_THRESHOLD, cmap='autumn', title=title, colorbar=True)
        display(viewer)
        plot_condition_bars(values, title)

selector.observe(render_result, names='value')
display(selector, output)
render_result()

## Interpretation note

The bars show participant-level means and SEM extracted from the stage-2 beta maps. Because the displayed region was selected from the same group contrast, these values are a descriptive visualization of the result, not an independent ROI test. Confirmatory ROI inference requires an independently defined mask or held-out data.